# 04 · 一条消息怎样跑完

假设你说「核查这份 ISO 9001 证书是否由投标人持有，且在投标截止日仍有效」。指令里只有任务要求。投标人、截止日和证书原文在附件材料里。这个案例要回答三个问题：**消息何时成为模型输入，模型怎么拿到证据，程序凭什么继续或停止。**



图中为简洁只画了一次读取。实际可能先搜索再读取，也可能在一次模型响应里请求多个工具。本章的 ISO 9001 轨迹是**教学数据**。

## 1 · 接收是一项数据库事实

Server 的 `session.prompt` handler 将请求交给 Core。Core 取得或生成 `messageID`，首次提交用 `SessionPrompt.prepare` 整理输入，再调用 `admission.admit`。后者发布 `InboxEnqueued`，投影在 `session_inbox` 中保存待交付记录。默认在这之后才调用 `execution.wake(sessionID)`。如果请求带 `resume: false`，这次提交只完成接收，稍后仍可由其它执行机会交付。[服务端入口](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/server/src/handlers/session.ts#L290) · [Admission](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/session.ts#L153) · [入队投影](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/projector.ts#L1040)

这个边界让「我接收过这条输入」和「模型已读到这条输入」可以分别核实。拿着同一个 `messageID`，你可以沿着待交付记录追到用户消息。

## 2 · 运行者决定什么时候交付

`SessionExecution` 按 Session 找到它所在的 Location 和 Runner。同一个 Session 的执行由协调器串行管理，额外 wake 可以合并。Runner 到了安全边界，先准备上下文，然后调用 `SessionInbox.promote`。普通对话路径下，`steer` 可在当前工作进行中的安全步骤边界交付，`queue` 则等到会话原本要进入空闲时再逐条交付。移动、压缩等控制项还有优先级，图里只画普通输入。[执行入口](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/execution.ts#L105) · [协调器](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/run-coordinator.ts#L135) · [交付选择](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/inbox.ts#L567)

`InboxDelivered` 的投影在同一事务里消费 inbox 行，并用原 ID 插入 `type: "user"` 的消息。随后才加载会话历史。这两个动作的顺序解释了为什么新输入可以出现在紧接着的模型请求里。[交付投影](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/projector.ts#L1011) · [先交付再读取](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/llm.ts#L191)

## 3 · Step 是一次逻辑模型调用

`runStep` 取到当前 Agent、模型、指令、工具和历史，组装主请求。一次尝试由 `step.ts` 发起一次 `llm.stream()`。重试由上层处理，因此同一个逻辑 Step 在出错后可能经历多次尝试。假设模型请求读取证书所在页，它发出工具名、参数与调用 ID。对于本地执行的调用，OpenCode 先发布调用事件，再执行工具，把结果与原 ID 一起记录到会话中。模型服务商已执行的工具走自己的结果路径。[主请求准备](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/llm.ts#L237) · [模型流和本地执行](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/step.ts#L137) · [本地结果发布](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/publish-llm-event.ts#L582)

下一步加载新历史，`to-llm-message.ts` 把存好的调用及其结果转换成请求消息。此时模型才获得证书持有人和有效日期，可以继续查投标人和截止日，或者作出有依据的结论。[调用与结果转换](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/to-llm-message.ts#L105)

## 4 · 局部步骤与整次会话有不同的完成条件

正常完成时，Step 会根据工具记录判断 `needsContinuation`。外层收到这个值后再次走安全边界。即使模型已经给出最终文本，仍可能存在待交付输入，或者 idle Hook 会追加工作。反过来，工具拒绝、执行中断、输出受限、重试和步数上限都走各自的分支。[续跑来源](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/publish-llm-event.ts#L615) · [空闲判断](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/llm.ts#L177) · [主循环](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/llm.ts#L216)



## 在 Notebook 里看两条轨迹

1. **原版测试的期望事件与录制响应。** 上游测试提交的是 `Say hello in one short sentence.`，带 `resume: false`，随后显式 resume。Notebook 从原版 TypeScript 测试提取事件断言，并逐帧打开原版录制的 HTTP 流。这里能分别看到「运行时预期记录什么」和「上游模型真实返回过什么」。[原版测试](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-runner-recorded.test.ts#L225)
2. **ISO 9001 教学轨迹。** 同一条核查指令经过接收、交付、调用工具、结果回传和最终作答。切换观察时点，看看 inbox、历史和工具结果中已经有了什么。教学数据与前面的真实 HTTP 录制分别标注。

**自测。** 如果你能看到证书的工具结果，但暂时没看到最终结论，系统可能已经完成了什么？根据前面的图，解释工具结果为什么要保留原调用 ID，以及下一步怎样拿到它。

In [1]:
from pathlib import Path
import json

# 从课程根目录或 notebooks/ 打开时均可读取随附资料。
HERE = Path.cwd().resolve()
COURSE = next(folder for folder in (HERE, *HERE.parents) if (folder / "fixtures" / "iso-9001.json").is_file())

## 观察 A · 原版测试断言与真实 HTTP 录制

左边的问题是运行时应记录哪些事件，右边的问题是模型服务曾返回什么。两份证据来自同一个上游测试案例。运行下面的观察单元不会向模型服务发请求。

In [2]:
import re

upstream = COURSE / "fixtures/upstream"
test = (upstream / "session-runner-recorded.test.ts").read_text()
start = test.index("session.inbox.enqueued.1")
end = test.index("session.step.ended.1", start) + len("session.step.ended.1")
events = re.findall(r"session\.[\w.-]+", test[start:end])
print("TypeScript 测试断言")
print(" → ".join(events))
assert events.index("session.inbox.delivered.1") < events.index("session.step.started.1")

cassette = json.loads((upstream / "openai-chat-streams-text.json").read_text())
interaction = cassette["interactions"][0]
wire = interaction["response"]["body"]
frames = [json.loads(line[6:]) for line in wire.splitlines()
          if line.startswith("data: ") and line != "data: [DONE]"]
choices = [choice for frame in frames for choice in frame["choices"]]
print("\n实际录制的模型请求和响应")
print("提交给模型的用户文本 ·", json.loads(interaction["request"]["body"])["messages"][-1]["content"])
print("响应内容 ·", "".join(choice["delta"].get("content", "") for choice in choices))
print("结束原因 ·", [choice["finish_reason"] for choice in choices if choice["finish_reason"]])
assert any(choice["finish_reason"] == "stop" for choice in choices)

TypeScript 测试断言
session.inbox.enqueued.1 → session.instructions.updated.2 → session.inbox.delivered.1 → session.step.started.1 → session.step.first-token.1 → session.text.started.1 → session.text.ended.1 → session.step.streamed.1 → session.step.ended.1

实际录制的模型请求和响应
提交给模型的用户文本 · Say hello in one short sentence.
响应内容 · Hello!
结束原因 · ['stop']


## 观察 B · 换成 ISO 9001 核查

这里使用独立的虚构案例。逐行对照时序图，指出同一个消息 ID 在什么时候离开 inbox，以及工具结果何时能进入下一次模型请求。

In [3]:
case = json.loads((COURSE / "fixtures/iso-9001.json").read_text())
print(case["provenance"])
print(case["request"])
print("\n材料")
for page, text in case["pages"].items():
    print(f"{page}  {text}")

print("\n观察时点")
for point in case["checkpoints"]:
    print(f'【{point["stage"]}】 inbox · {point["inbox"]}')
    print(f'  历史 · {point["history"]}')
    print(f'  模型 · {point["visible_to_model"]}')

虚构教学案例。步骤顺序用于对照源码，未调用在线模型。
核对投标人是否提交了本公司持有且在投标截止日仍有效的 ISO 9001 证书。

材料
P04  投标人：华宁设备服务有限公司。
P07  投标截止日期：2026-11-25。
P12  ISO 9001 质量管理体系认证。证书持有人：华宁设备服务有限公司。有效期至 2027-06-30。

观察时点
【接收】 inbox · msg_iso_9001
  历史 · 尚无用户消息
  模型 · 尚未发出核查请求
【交付】 inbox · 已消费
  历史 · user: msg_iso_9001
  模型 · 下一步可读取用户指令
【工具返回】 inbox · 已消费
  历史 · 用户指令、调用 call_read_p12、同 ID 的结果
  模型 · 下次请求带证书原文，可继续核对主体与截止日
【作答】 inbox · 已消费
  历史 · 三页证据、核查结论
  模型 · 当前步骤已收尾，Runner 再检查待交付输入


## 观察 C · 可选运行原版 Core 集成测试

若已有按锁文件安装完成的原版工作区，可在启动 Jupyter 时设置环境变量 LEARN_OPENCODE_SOURCE 指向其目录。下面会验证提交并由原版测试入口重放 HTTP 录制。

In [4]:
import os
import shutil
import subprocess

source = os.environ.get("LEARN_OPENCODE_SOURCE")
if source is None:
    print("默认离线观察模式。设置 LEARN_OPENCODE_SOURCE 后可运行原版集成测试。")
else:
    repo = Path(source).expanduser().resolve()
    pinned = "ee5fe34e136488a1839ac73bcca8321c51a31b75"
    actual = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
    if actual != pinned:
        raise RuntimeError("请使用课程固定提交的独立副本")
    bun = shutil.which("bun")
    if bun is None:
        raise RuntimeError("请先在 PATH 中提供仓库声明的 Bun")
    environment = {name: value for name, value in os.environ.items() if name != "RECORD"}
    environment["CI"] = "true"
    result = subprocess.run(
        [bun, "run", "test", "test/session-runner-recorded.test.ts"],
        cwd=repo / "packages/core", env=environment, capture_output=True, text=True, timeout=30,
    )
    print(result.stdout, result.stderr, sep="")
    result.check_returncode()


bun test v1.4.4 (38afee6c8)
$ bun run script/test.ts test/session-runner-recorded.test.ts

 1 pass
 0 fail
 13 expect() calls
Ran 1 test across 1 file. [501.00ms]

